# ✈️ Travel & Tourism ML Project — End-to-End MLOps
> **Capstone Project** | Regression · Classification · Recommendation · Flask API · Docker · Kubernetes · Airflow · Jenkins · MLflow · Streamlit

---
## 📋 Table of Contents
| # | Objective |
|---|---|
| 1 | 📦 Setup & Data Upload |
| 2 | 🔍 EDA — Exploratory Data Analysis |
| 3 | ✈️ Regression Model — Flight Price Prediction |
| 4 | 🌐 REST API with Flask (ngrok tunnel) |
| 5 | 🐳 Containerization with Docker |
| 6 | ☸️ Kubernetes Deployment |
| 7 | 🌬️ Apache Airflow DAGs |
| 8 | 🔧 CI/CD Pipeline with Jenkins |
| 9 | 📊 Model Tracking with MLflow |
| 10 | 🚻 Gender Classification Model |
| 11 | 🏨 Travel Recommendation + Streamlit App |

---
## 📦 SECTION 1 — Setup & Data Upload

In [ ]:
# ── Install all required libraries ──────────────────────────────────────────
!pip install -q mlflow scikit-learn flask flask-restful pyngrok joblib
!pip install -q streamlit pyngrok
!pip install -q imbalanced-learn
print('✅ All packages installed')

In [ ]:
# ── Core imports ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, os, json, joblib, re
from datetime import datetime

from sklearn.model_selection  import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing    import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline         import Pipeline
from sklearn.compose          import ColumnTransformer
from sklearn.metrics          import (mean_absolute_error, mean_squared_error,
                                       r2_score, accuracy_score,
                                       classification_report, confusion_matrix)
from sklearn.ensemble         import (RandomForestRegressor, GradientBoostingRegressor,
                                       HistGradientBoostingRegressor,
                                       HistGradientBoostingClassifier,
                                       RandomForestClassifier, GradientBoostingClassifier)
from sklearn.linear_model     import LinearRegression, LogisticRegression
from sklearn.neighbors        import NearestNeighbors

import mlflow
import mlflow.sklearn

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

os.makedirs('models',  exist_ok=True)
os.makedirs('outputs', exist_ok=True)
print('✅ Imports complete')

In [ ]:
# ── Upload your CSV files (flights.csv, hotels.csv, users.csv) ───────────────
from google.colab import files
print('📂 Please upload: flights.csv, hotels.csv, users.csv')
uploaded = files.upload()
print('\n✅ Uploaded files:', list(uploaded.keys()))

In [ ]:
# ── Load datasets ─────────────────────────────────────────────────────────────
flights_df = pd.read_csv('flights.csv')
hotels_df  = pd.read_csv('hotels.csv')
users_df   = pd.read_csv('users.csv')

# Parse dates
flights_df['date'] = pd.to_datetime(flights_df['date'], dayfirst=False)
hotels_df['date']  = pd.to_datetime(hotels_df['date'],  dayfirst=False)

print(f'✈️  Flights : {flights_df.shape}')
print(f'🏨  Hotels  : {hotels_df.shape}')
print(f'👤  Users   : {users_df.shape}')
flights_df.head(3)

---
## 🔍 SECTION 2 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Basic statistics ──────────────────────────────────────────────────────────
print('='*55)
print('FLIGHTS — describe')
print('='*55)
display(flights_df.describe().round(2))

print('\n' + '='*55)
print('Missing values')
print('='*55)
for name, df in [('flights', flights_df), ('hotels', hotels_df), ('users', users_df)]:
    print(f'  {name}: {df.isnull().sum().sum()} missing values')

In [ ]:
# ── EDA Plot 1 — Flight price distribution by class ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Price distribution
axes[0].hist(flights_df['price'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Flight Price Distribution', fontsize=13)
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

# Price by flight type
flights_df.boxplot(column='price', by='flightType', ax=axes[1])
axes[1].set_title('Price by Flight Type', fontsize=13)
axes[1].set_xlabel('Flight Type')
axes[1].set_ylabel('Price ($)')
plt.sca(axes[1]); plt.xticks(rotation=15)

# Agency count
flights_df['agency'].value_counts().plot(kind='bar', ax=axes[2],
                                          color=['steelblue','coral','seagreen'])
axes[2].set_title('Bookings per Agency', fontsize=13)
axes[2].set_xlabel('Agency')
axes[2].set_ylabel('Count')
plt.xticks(rotation=15)

plt.suptitle('')
plt.tight_layout()
plt.savefig('outputs/eda_flights.png', bbox_inches='tight')
plt.show()
print('✅ EDA plot 1 saved')

In [ ]:
# ── EDA Plot 2 — Correlation, distance vs price, user gender ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Correlation heatmap (numeric cols)
num_cols = ['price', 'time', 'distance']
corr = flights_df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Feature Correlation', fontsize=13)

# Distance vs Price
sample = flights_df.sample(3000, random_state=42)
colors = {'economic': 'steelblue', 'premium': 'orange', 'firstClass': 'crimson'}
for ft, grp in sample.groupby('flightType'):
    axes[1].scatter(grp['distance'], grp['price'], alpha=0.3, s=8,
                    label=ft, color=colors.get(ft, 'gray'))
axes[1].set_title('Distance vs Price', fontsize=13)
axes[1].set_xlabel('Distance (km)')
axes[1].set_ylabel('Price ($)')
axes[1].legend(fontsize=8)

# User gender distribution
gender_counts = users_df['gender'].value_counts()
axes[2].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#4C72B0','#DD8452'], startangle=90)
axes[2].set_title('User Gender Distribution', fontsize=13)

plt.tight_layout()
plt.savefig('outputs/eda_correlations.png', bbox_inches='tight')
plt.show()
print('✅ EDA plot 2 saved')

In [ ]:
# ── EDA Plot 3 — Hotels analysis ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Top 10 hotel places
top_places = hotels_df['place'].value_counts().head(10)
top_places.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 10 Hotel Destinations', fontsize=13)
axes[0].set_xlabel('Bookings')

# Hotel price distribution
axes[1].hist(hotels_df['price'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Hotel Price/Night Distribution', fontsize=13)
axes[1].set_xlabel('Price per Night ($)')
axes[1].set_ylabel('Count')

# Days stayed distribution
hotels_df['days'].value_counts().sort_index().plot(kind='bar', ax=axes[2],
                                                    color='seagreen')
axes[2].set_title('Hotel Stay Duration', fontsize=13)
axes[2].set_xlabel('Days')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('outputs/eda_hotels.png', bbox_inches='tight')
plt.show()
print('✅ EDA plot 3 saved')

---
## ✈️ SECTION 3 — Regression Model: Flight Price Prediction
> **Objective 1** — Build a regression model using `flights.csv` to predict flight price.

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────────────────
df_flight = flights_df.copy()

# Extract date features
df_flight['month']      = df_flight['date'].dt.month
df_flight['dayofweek']  = df_flight['date'].dt.dayofweek
df_flight['year']       = df_flight['date'].dt.year

# Encode categorical
le_ft  = LabelEncoder()
le_ag  = LabelEncoder()
le_fr  = LabelEncoder()
le_to  = LabelEncoder()

df_flight['flightType_enc'] = le_ft.fit_transform(df_flight['flightType'])
df_flight['agency_enc']     = le_ag.fit_transform(df_flight['agency'])
df_flight['from_enc']       = le_fr.fit_transform(df_flight['from'])
df_flight['to_enc']         = le_to.fit_transform(df_flight['to'])

FEATURES = ['time', 'distance', 'flightType_enc', 'agency_enc',
            'from_enc', 'to_enc', 'month', 'dayofweek', 'year']
TARGET   = 'price'

X = df_flight[FEATURES]
y = df_flight[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'Training samples : {X_train.shape[0]:,}')
print(f'Test samples     : {X_test.shape[0]:,}')
print(f'Features         : {FEATURES}')

In [ ]:
# ── Train & Compare Multiple Regression Models ────────────────────────────────
# ⚡ SPEED OPTIMIZATIONS:
#   • RandomForest  : n_estimators=50, max_samples=0.6 (trains on 60% of rows per tree)
#   • HistGradientBoosting: sklearn's fast histogram-based GB (10-50x faster than GradientBoostingRegressor)
#   • LinearRegression: unchanged (already instant)
from sklearn.ensemble import HistGradientBoostingRegressor

reg_models = {
    'Linear Regression':       LinearRegression(),
    'Random Forest':           RandomForestRegressor(
                                   n_estimators=50,       # ⚡ 50 instead of 100
                                   max_depth=15,          # ⚡ cap depth
                                   max_samples=0.6,       # ⚡ subsample rows per tree
                                   n_jobs=-1, random_state=42),
    'Hist Gradient Boosting':  HistGradientBoostingRegressor(
                                   max_iter=100,          # ⚡ histogram-based = 10-50x faster
                                   max_depth=8,
                                   random_state=42),
}

import time
reg_results = {}
for name, model in reg_models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    reg_results[name] = {
        'MAE':  round(mean_absolute_error(y_test, preds), 2),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, preds)), 2),
        'R2':   round(r2_score(y_test, preds), 4),
    }
    print(f'{name:28s} | MAE={reg_results[name]["MAE"]:8.2f} '
          f'| RMSE={reg_results[name]["RMSE"]:8.2f} '
          f'| R2={reg_results[name]["R2"]:.4f}  ({time.time()-t0:.1f}s)')

best_reg_name = max(reg_results, key=lambda k: reg_results[k]['R2'])
best_reg_model = reg_models[best_reg_name]
print(f'\n🏆 Best model: {best_reg_name} (R2={reg_results[best_reg_name]["R2"]})')

In [ ]:
# ── ⚡ Fast Hyperparameter Tuning with RandomizedSearchCV ─────────────────────
# SPEED TRICKS APPLIED:
#   1. RandomizedSearchCV  → samples only n_iter=10 combos (vs GridSearch = all 36)
#   2. cv=2                → 2-fold instead of 3 (33% fewer fits)
#   3. Subsample 40K rows  → tunes on representative subset, not all 217K train rows
#   4. n_jobs=-1           → uses all CPU cores in parallel
#   5. Tight search space  → no None depth, only fast-converging values
from sklearn.model_selection import RandomizedSearchCV
import time

# ── Subsample for tuning (fast, representative) ───────────────────────────────
TUNE_SIZE = 40_000
idx_tune  = np.random.choice(len(X_train), size=min(TUNE_SIZE, len(X_train)), replace=False)
X_tune    = X_train.iloc[idx_tune]
y_tune    = y_train.iloc[idx_tune]
print(f'⚡ Tuning on {len(X_tune):,} rows (subsample) instead of {len(X_train):,}')

param_dist = {
    'n_estimators':      [50, 100, 150],
    'max_depth':         [8, 12, 15, 20],
    'min_samples_split': [2, 5, 10],
    'max_features':      ['sqrt', 0.5],
    'max_samples':       [0.6, 0.8],
}

t0 = time.time()
rand_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions = param_dist,
    n_iter   = 10,        # ⚡ only 10 random combos
    cv       = 2,         # ⚡ 2-fold (fast)
    scoring  = 'r2',
    n_jobs   = -1,        # ⚡ all CPU cores
    verbose  = 1,
    random_state = 42,
)
rand_search.fit(X_tune, y_tune)
print(f'\n⏱️  Tuning completed in {time.time()-t0:.1f}s')

# ── Refit best params on FULL training set ────────────────────────────────────
best_params = rand_search.best_params_
best_rf = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
best_rf.fit(X_train, y_train)   # final fit on all training data
preds_tuned = best_rf.predict(X_test)

print(f'\nBest params : {best_params}')
print(f'Tuned MAE   : {mean_absolute_error(y_test, preds_tuned):.2f}')
print(f'Tuned RMSE  : {np.sqrt(mean_squared_error(y_test, preds_tuned)):.2f}')
print(f'Tuned R2    : {r2_score(y_test, preds_tuned):.4f}')

In [ ]:
# ── Regression Evaluation Plots ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Actual vs Predicted
sample_idx = np.random.choice(len(y_test), 800, replace=False)
y_sample   = np.array(y_test)[sample_idx]
p_sample   = preds_tuned[sample_idx]
axes[0].scatter(y_sample, p_sample, alpha=0.3, s=10, color='steelblue')
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect fit')
axes[0].set_title('Actual vs Predicted Price', fontsize=13)
axes[0].set_xlabel('Actual ($)'); axes[0].set_ylabel('Predicted ($)')
axes[0].legend(fontsize=9)

# 2. Residuals distribution
residuals = y_test.values - preds_tuned
axes[1].hist(residuals, bins=60, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', lw=1.5, ls='--')
axes[1].set_title('Residuals Distribution', fontsize=13)
axes[1].set_xlabel('Residual ($)'); axes[1].set_ylabel('Count')

# 3. Feature importance
importances = pd.Series(best_rf.feature_importances_, index=FEATURES)
importances.sort_values().plot(kind='barh', ax=axes[2], color='steelblue')
axes[2].set_title('Feature Importances', fontsize=13)
axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('outputs/regression_evaluation.png', bbox_inches='tight')
plt.show()
print('✅ Regression evaluation plot saved')

In [ ]:
# ── Save Regression Model & Encoders ─────────────────────────────────────────
flight_model_artifacts = {
    'model':    best_rf,
    'le_ft':    le_ft,
    'le_ag':    le_ag,
    'le_fr':    le_fr,
    'le_to':    le_to,
    'features': FEATURES,
}
joblib.dump(flight_model_artifacts, 'models/flight_price_model.pkl')
print('✅ Flight price model saved → models/flight_price_model.pkl')

---
## 🌐 SECTION 4 — REST API with Flask
> **Objective 2** — Serve the flight price prediction model via a REST API.

In [ ]:
# ── Write the Flask application ───────────────────────────────────────────────
flask_app_code = '''
from flask import Flask, request, jsonify
import numpy as np
import joblib

app = Flask(__name__)

# Load model artifacts
artifacts = joblib.load("models/flight_price_model.pkl")
model     = artifacts["model"]
le_ft     = artifacts["le_ft"]
le_ag     = artifacts["le_ag"]
le_fr     = artifacts["le_fr"]
le_to     = artifacts["le_to"]


@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "status": "running",
        "endpoints": {
            "predict_price":  "POST /predict/flight-price",
            "health":         "GET  /health"
        }
    })


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy", "model": "RandomForestRegressor"})


@app.route("/predict/flight-price", methods=["POST"])
def predict_flight_price():
    """
    Expects JSON:
    {
      "time": 2.5,
      "distance": 950.0,
      "flightType": "economic",
      "agency": "FlyingDrops",
      "from": "Recife (PE)",
      "to": "Sao Paulo (SP)",
      "month": 6,
      "dayofweek": 2,
      "year": 2023
    }
    """
    try:
        data = request.get_json(force=True)

        # Encode categoricals (handle unseen labels gracefully)
        def safe_transform(le, val):
            if val in le.classes_:
                return int(le.transform([val])[0])
            return -1   # unknown category

        features = np.array([[
            float(data["time"]),
            float(data["distance"]),
            safe_transform(le_ft, data["flightType"]),
            safe_transform(le_ag, data["agency"]),
            safe_transform(le_fr, data["from"]),
            safe_transform(le_to, data["to"]),
            int(data.get("month",      6)),
            int(data.get("dayofweek",  2)),
            int(data.get("year",    2023)),
        ]])

        prediction = float(model.predict(features)[0])
        return jsonify({
            "predicted_price_usd": round(prediction, 2),
            "input": data,
            "status": "success"
        })

    except Exception as e:
        return jsonify({"error": str(e), "status": "failed"}), 400


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

with open('app.py', 'w') as f:
    f.write(flask_app_code)
print('✅ Flask app written → app.py')

In [ ]:
# ── Start Flask + expose via ngrok ────────────────────────────────────────────
# NOTE: Get a free auth token at https://ngrok.com → Your Authtoken
# Replace the token below with your own before running

import subprocess, threading, time
from pyngrok import ngrok, conf

NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"   # <── replace this

if NGROK_TOKEN != "PASTE_YOUR_NGROK_TOKEN_HERE":
    # Set token
    !ngrok config add-authtoken {NGROK_TOKEN}

    # Launch Flask in background thread
    def run_flask():
        subprocess.run(['python', 'app.py'])

    flask_thread = threading.Thread(target=run_flask, daemon=True)
    flask_thread.start()
    time.sleep(3)

    # Open ngrok tunnel
    public_url = ngrok.connect(5000).public_url
    print(f'\n🌐 Flask API is live at: {public_url}')
    print(f'   Health check : {public_url}/health')
    print(f'   Predict      : POST {public_url}/predict/flight-price')
else:
    print('⚠️  Add your ngrok token to start the live API.')
    print('   (API code is ready — just fill in the token)')

In [ ]:
# ── Test the API locally (no ngrok needed) ────────────────────────────────────
import joblib, numpy as np

def local_predict_flight(time, distance, flightType, agency,
                          from_city, to_city, month=6, dayofweek=2, year=2023):
    arts = joblib.load('models/flight_price_model.pkl')
    mdl  = arts['model']

    def safe(le, v):
        return int(le.transform([v])[0]) if v in le.classes_ else -1

    X = np.array([[time, distance,
                   safe(arts['le_ft'], flightType),
                   safe(arts['le_ag'], agency),
                   safe(arts['le_fr'], from_city),
                   safe(arts['le_to'], to_city),
                   month, dayofweek, year]])
    return round(float(mdl.predict(X)[0]), 2)

# --- DEMO PREDICTIONS ---
test_cases = [
    (2.5,  950.0,  'economic',   'FlyingDrops', 'Recife (PE)',    'Sao Paulo (SP)'),
    (5.0,  3200.0, 'firstClass', 'CloudFy',     'Brasilia (DF)',  'Rio de Janeiro (RJ)'),
    (1.2,  400.0,  'premium',    'Rainbow',     'Natal (RN)',     'Florianopolis (SC)'),
]
print(f'{"Route":<42} {"Type":<12} {"Agency":<14} {"Predicted Price":>16}')
print('-'*85)
for t, d, ft, ag, fr, to_ in test_cases:
    p = local_predict_flight(t, d, ft, ag, fr, to_)
    print(f'{fr} → {to_:<25} {ft:<12} {ag:<14} ${p:>14,.2f}')

---
## 🐳 SECTION 5 — Containerization with Docker
> **Objective 3** — Package the model and Flask API into a Docker container.

The cells below **generate all Docker files** and show you the exact commands to build and run.

In [ ]:
# ── Generate Dockerfile ───────────────────────────────────────────────────────
dockerfile = """
# ─── Dockerfile ───────────────────────────────────────────────────────────────
FROM python:3.10-slim

# Set working directory
WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    build-essential && rm -rf /var/lib/apt/lists/*

# Copy requirements and install
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code and model
COPY app.py .
COPY models/ models/

# Expose Flask port
EXPOSE 5000

# Health-check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \\
    CMD curl -f http://localhost:5000/health || exit 1

# Start the API
CMD ["python", "app.py"]
"""

requirements_txt = """flask==3.0.2
scikit-learn==1.4.1
numpy==1.26.4
joblib==1.3.2
gunicorn==21.2.0
"""

dockerignore = """__pycache__/
*.pyc
*.pyo
.env
outputs/
"""

with open('Dockerfile',        'w') as f: f.write(dockerfile)
with open('requirements.txt',  'w') as f: f.write(requirements_txt)
with open('.dockerignore',     'w') as f: f.write(dockerignore)

print('✅ Docker files created:')
print('   📄 Dockerfile')
print('   📄 requirements.txt')
print('   📄 .dockerignore')
print()
print('──── Commands to build & run (run in your terminal) ────')
print('  docker build -t travel-flight-api:v1 .')
print('  docker run  -d -p 5000:5000 --name flight-api travel-flight-api:v1')
print('  docker logs flight-api')
print('  curl http://localhost:5000/health')

In [ ]:
# ── docker-compose.yml (multi-service) ────────────────────────────────────────
docker_compose = """version: '3.9'

services:
  flight-api:
    build: .
    image: travel-flight-api:v1
    container_name: flight_price_api
    ports:
      - "5000:5000"
    volumes:
      - ./models:/app/models:ro
    environment:
      - FLASK_ENV=production
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:5000/health"]
      interval: 30s
      timeout: 10s
      retries: 3

  mlflow:
    image: ghcr.io/mlflow/mlflow:v2.11.1
    container_name: mlflow_server
    ports:
      - "5001:5000"
    command: >
      mlflow server
      --host 0.0.0.0
      --port 5000
      --backend-store-uri sqlite:///mlflow.db
      --default-artifact-root /mlflow/artifacts
    volumes:
      - mlflow_data:/mlflow
    restart: unless-stopped

volumes:
  mlflow_data:
"""

with open('docker-compose.yml', 'w') as f:
    f.write(docker_compose)

print('✅ docker-compose.yml created')
print()
print('──── Commands ────')
print('  docker-compose up -d        # start all services')
print('  docker-compose down         # stop all services')
print('  docker-compose ps           # service status')

---
## ☸️ SECTION 6 — Kubernetes Deployment
> **Objective 4** — Deploy and scale the model using Kubernetes manifests.

In [ ]:
# ── Generate all Kubernetes YAML manifests ────────────────────────────────────
import os
os.makedirs('k8s', exist_ok=True)

# 1. Deployment
deployment_yaml = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: flight-api-deployment
  labels:
    app: flight-price-api
spec:
  replicas: 3
  selector:
    matchLabels:
      app: flight-price-api
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 0
  template:
    metadata:
      labels:
        app: flight-price-api
    spec:
      containers:
      - name: flight-api
        image: travel-flight-api:v1
        ports:
        - containerPort: 5000
        resources:
          requests:
            memory: "256Mi"
            cpu:    "250m"
          limits:
            memory: "512Mi"
            cpu:    "500m"
        readinessProbe:
          httpGet:
            path: /health
            port: 5000
          initialDelaySeconds: 5
          periodSeconds: 10
        livenessProbe:
          httpGet:
            path: /health
            port: 5000
          initialDelaySeconds: 15
          periodSeconds: 20
"""

# 2. Service
service_yaml = """apiVersion: v1
kind: Service
metadata:
  name: flight-api-service
spec:
  type: LoadBalancer
  selector:
    app: flight-price-api
  ports:
  - protocol: TCP
    port: 80
    targetPort: 5000
"""

# 3. Horizontal Pod Autoscaler
hpa_yaml = """apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: flight-api-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: flight-api-deployment
  minReplicas: 2
  maxReplicas: 10
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 60
"""

# 4. ConfigMap
configmap_yaml = """apiVersion: v1
kind: ConfigMap
metadata:
  name: flight-api-config
data:
  FLASK_ENV: "production"
  MODEL_PATH: "/app/models/flight_price_model.pkl"
  LOG_LEVEL: "INFO"
"""

files_to_write = {
    'k8s/deployment.yaml':  deployment_yaml,
    'k8s/service.yaml':     service_yaml,
    'k8s/hpa.yaml':         hpa_yaml,
    'k8s/configmap.yaml':   configmap_yaml,
}

for path, content in files_to_write.items():
    with open(path, 'w') as f:
        f.write(content)

print('✅ Kubernetes manifests generated in k8s/')
for p in files_to_write:
    print(f'   📄 {p}')
print()
print('──── Deploy commands (kubectl) ────')
print('  kubectl apply -f k8s/configmap.yaml')
print('  kubectl apply -f k8s/deployment.yaml')
print('  kubectl apply -f k8s/service.yaml')
print('  kubectl apply -f k8s/hpa.yaml')
print('  kubectl get pods -w                     # watch pods')
print('  kubectl scale deployment flight-api-deployment --replicas=5')

---
## 🌬️ SECTION 7 — Apache Airflow DAGs
> **Objective 5** — Automate model training & data pipelines with Airflow DAGs.

In [ ]:
# ── Generate Airflow DAGs ─────────────────────────────────────────────────────
import os
os.makedirs('dags', exist_ok=True)

dag_code = '''
"""
Apache Airflow DAG — Travel ML Pipeline
Orchestrates: data validation → preprocessing → model training → evaluation → deployment
"""

from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash   import BashOperator
from airflow.utils.dates      import days_ago
import pandas as pd
import numpy as np
import joblib
import logging

# ── Default DAG arguments ─────────────────────────────────────────────────────
default_args = {
    "owner":            "data_team",
    "depends_on_past":  False,
    "email":            ["alerts@travelml.com"],
    "email_on_failure": True,
    "email_on_retry":   False,
    "retries":          2,
    "retry_delay":      timedelta(minutes=5),
}

# ── DAG definition ────────────────────────────────────────────────────────────
with DAG(
    dag_id             = "travel_ml_pipeline",
    default_args       = default_args,
    description        = "End-to-end ML pipeline for Travel price prediction",
    schedule_interval  = "0 2 * * *",   # runs daily at 02:00
    start_date         = days_ago(1),
    catchup            = False,
    tags               = ["travel", "ml", "regression"],
) as dag:

    # ── Task 1: Validate raw data ─────────────────────────────────────────────
    def validate_data(**context):
        logging.info("Validating input datasets...")
        flights = pd.read_csv("/data/flights.csv")
        hotels  = pd.read_csv("/data/hotels.csv")
        users   = pd.read_csv("/data/users.csv")

        assert not flights.empty, "flights.csv is empty!"
        assert not hotels.empty,  "hotels.csv is empty!"
        assert not users.empty,   "users.csv is empty!"
        assert "price" in flights.columns, "Missing price column in flights!"

        # Push stats to XCom
        context["ti"].xcom_push("flight_rows", len(flights))
        context["ti"].xcom_push("hotel_rows",  len(hotels))
        logging.info(f"Validation passed — {len(flights)} flight records.")

    t1_validate = PythonOperator(
        task_id         = "validate_data",
        python_callable = validate_data,
        provide_context = True,
    )

    # ── Task 2: Preprocess data ────────────────────────────────────────────────
    def preprocess_data(**context):
        from sklearn.preprocessing import LabelEncoder
        import joblib

        logging.info("Preprocessing data...")
        df = pd.read_csv("/data/flights.csv")
        df["date"] = pd.to_datetime(df["date"])
        df["month"]     = df["date"].dt.month
        df["dayofweek"] = df["date"].dt.dayofweek
        df["year"]      = df["date"].dt.year

        for col in ["flightType", "agency", "from", "to"]:
            le = LabelEncoder()
            df[f"{col}_enc"] = le.fit_transform(df[col])
            joblib.dump(le, f"/models/le_{col}.pkl")

        df.to_parquet("/data/flights_processed.parquet", index=False)
        logging.info("Preprocessing complete.")

    t2_preprocess = PythonOperator(
        task_id         = "preprocess_data",
        python_callable = preprocess_data,
        provide_context = True,
    )

    # ── Task 3: Train model ────────────────────────────────────────────────────
    def train_model(**context):
        from sklearn.ensemble    import RandomForestRegressor
        from sklearn.model_selection import train_test_split
        from sklearn.metrics     import r2_score
        import mlflow, mlflow.sklearn

        logging.info("Training model...")
        df = pd.read_parquet("/data/flights_processed.parquet")

        FEATURES = ["time","distance","flightType_enc",
                    "agency_enc","from_enc","to_enc",
                    "month","dayofweek","year"]
        X = df[FEATURES]; y = df["price"]
        X_tr, X_ts, y_tr, y_ts = train_test_split(X, y, test_size=0.2, random_state=42)

        with mlflow.start_run(run_name="airflow_daily_train"):
            model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
            model.fit(X_tr, y_tr)
            r2 = r2_score(y_ts, model.predict(X_ts))
            mlflow.log_param("n_estimators", 100)
            mlflow.log_metric("r2_score", r2)
            mlflow.sklearn.log_model(model, "flight_price_model")
            joblib.dump(model, "/models/flight_price_model.pkl")
            logging.info(f"Model trained — R2={r2:.4f}")
            context["ti"].xcom_push("r2_score", r2)

    t3_train = PythonOperator(
        task_id         = "train_model",
        python_callable = train_model,
        provide_context = True,
    )

    # ── Task 4: Evaluate & gate ────────────────────────────────────────────────
    def evaluate_model(**context):
        r2 = context["ti"].xcom_pull(task_ids="train_model", key="r2_score")
        logging.info(f"Model R2: {r2}")
        if r2 < 0.70:
            raise ValueError(f"Model R2={r2:.4f} below threshold 0.70 — deployment blocked!")
        logging.info("Model passed quality gate — proceeding to deployment.")

    t4_evaluate = PythonOperator(
        task_id         = "evaluate_model",
        python_callable = evaluate_model,
        provide_context = True,
    )

    # ── Task 5: Deploy (restart Docker container) ─────────────────────────────
    t5_deploy = BashOperator(
        task_id      = "deploy_model",
        bash_command = (
            "docker build -t travel-flight-api:latest . && "
            "docker stop flight_price_api || true && "
            "docker rm   flight_price_api || true && "
            "docker run -d -p 5000:5000 --name flight_price_api "
            "           -v $(pwd)/models:/app/models travel-flight-api:latest"
        ),
    )

    # ── Task 6: Notify ────────────────────────────────────────────────────────
    t6_notify = BashOperator(
        task_id      = "notify_success",
        bash_command = "echo '✅ Travel ML pipeline completed successfully!'",
    )

    # ── DAG dependency graph ───────────────────────────────────────────────────
    t1_validate >> t2_preprocess >> t3_train >> t4_evaluate >> t5_deploy >> t6_notify
'''

with open('dags/travel_ml_pipeline.py', 'w') as f:
    f.write(dag_code)

print('✅ Airflow DAG written → dags/travel_ml_pipeline.py')
print()
print('──── Airflow setup commands ────')
print('  pip install apache-airflow')
print('  export AIRFLOW_HOME=~/airflow')
print('  airflow db init')
print('  cp dags/travel_ml_pipeline.py ~/airflow/dags/')
print('  airflow webserver --port 8080 &')
print('  airflow scheduler &')
print('  # Visit http://localhost:8080  (user: admin)')

In [ ]:
# ── Visualize the DAG structure ───────────────────────────────────────────────
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(16, 3.5))
ax.set_xlim(0, 18); ax.set_ylim(0, 4); ax.axis('off')
ax.set_title('Apache Airflow DAG — Travel ML Pipeline', fontsize=14, fontweight='bold', pad=12)

tasks = [
    (1.0,  'validate_data',    '#4C72B0'),
    (4.0,  'preprocess_data',  '#55A868'),
    (7.0,  'train_model',      '#C44E52'),
    (10.0, 'evaluate_model',   '#8172B2'),
    (13.0, 'deploy_model',     '#CCB974'),
    (16.0, 'notify_success',   '#64B5CD'),
]

for i, (x, label, color) in enumerate(tasks):
    rect = mpatches.FancyBboxPatch((x - 1.2, 1.2), 2.4, 1.6,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='white',
                                    linewidth=1.5, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x, 2.0, label.replace('_', '\n'), ha='center', va='center',
            fontsize=7.5, color='white', fontweight='bold')
    if i < len(tasks) - 1:
        ax.annotate('', xy=(tasks[i+1][0]-1.2, 2.0), xytext=(x+1.2, 2.0),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=1.8))

ax.text(9, 0.4, 'schedule: daily @ 02:00  |  retries: 2  |  email on failure',
        ha='center', fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.savefig('outputs/airflow_dag.png', bbox_inches='tight', dpi=120)
plt.show()
print('✅ DAG diagram saved')

---
## 🔧 SECTION 8 — CI/CD Pipeline with Jenkins
> **Objective 6** — Continuous Integration & Deployment pipeline for the travel model.

In [ ]:
# ── Generate Jenkinsfile ──────────────────────────────────────────────────────
jenkinsfile = """
// ── Jenkinsfile — Travel ML CI/CD Pipeline ───────────────────────────────────
pipeline {
    agent any

    environment {
        IMAGE_NAME   = "travel-flight-api"
        IMAGE_TAG    = "${BUILD_NUMBER}"
        REGISTRY     = "your-dockerhub-username"
        KUBECONFIG   = credentials('kubeconfig-secret')
    }

    stages {

        stage('🔁 Checkout') {
            steps {
                checkout scm
                echo "Branch: ${env.GIT_BRANCH} | Commit: ${env.GIT_COMMIT[0..7]}"
            }
        }

        stage('📦 Install Dependencies') {
            steps {
                sh 'pip install -r requirements.txt --quiet'
            }
        }

        stage('🧪 Run Tests') {
            steps {
                sh 'python -m pytest tests/ -v --tb=short'
            }
            post {
                always { junit 'test-results/*.xml' }
            }
        }

        stage('📊 Train & Evaluate Model') {
            steps {
                sh 'python scripts/train_pipeline.py'
            }
        }

        stage('🐳 Build Docker Image') {
            steps {
                script {
                    docker.build("${REGISTRY}/${IMAGE_NAME}:${IMAGE_TAG}")
                    docker.build("${REGISTRY}/${IMAGE_NAME}:latest")
                }
            }
        }

        stage('🔒 Security Scan') {
            steps {
                sh "docker run --rm -v /var/run/docker.sock:/var/run/docker.sock \\
                       aquasec/trivy image --exit-code 0 --severity HIGH,CRITICAL \\
                       ${REGISTRY}/${IMAGE_NAME}:${IMAGE_TAG}"
            }
        }

        stage('📤 Push to Registry') {
            when { branch 'main' }
            steps {
                script {
                    docker.withRegistry('https://registry.hub.docker.com',
                                        'dockerhub-credentials') {
                        docker.image("${REGISTRY}/${IMAGE_NAME}:${IMAGE_TAG}").push()
                        docker.image("${REGISTRY}/${IMAGE_NAME}:latest").push()
                    }
                }
            }
        }

        stage('🚀 Deploy to Kubernetes') {
            when { branch 'main' }
            steps {
                sh """
                    kubectl set image deployment/flight-api-deployment \\
                        flight-api=${REGISTRY}/${IMAGE_NAME}:${IMAGE_TAG}
                    kubectl rollout status deployment/flight-api-deployment
                    kubectl get pods -l app=flight-price-api
                """
            }
        }

        stage('✅ Smoke Test') {
            when { branch 'main' }
            steps {
                sh 'sleep 10 && curl -f http://flight-api-service/health'
            }
        }
    }

    post {
        success {
            echo '✅ Pipeline succeeded — model deployed!'
            slackSend(channel: '#ml-deployments',
                      color: 'good',
                      message: "✅ Travel ML deployed: Build #${BUILD_NUMBER}")
        }
        failure {
            echo '❌ Pipeline failed!'
            slackSend(channel: '#ml-deployments',
                      color: 'danger',
                      message: "❌ Travel ML pipeline FAILED: Build #${BUILD_NUMBER}")
            emailext(
                subject: "Jenkins Build FAILED — #${BUILD_NUMBER}",
                body: 'Check Jenkins for details.',
                to: 'team@travelml.com'
            )
        }
        always {
            cleanWs()
        }
    }
}
"""

with open('Jenkinsfile', 'w') as f:
    f.write(jenkinsfile)

print('✅ Jenkinsfile created')
print()
print('──── Jenkins setup steps ────')
print('1.  docker run -d -p 8080:8080 jenkins/jenkins:lts')
print('2.  Install plugins: Pipeline, Docker, Kubernetes, Slack, Git')
print('3.  Add credentials: dockerhub-credentials, kubeconfig-secret')
print('4.  New Item → Pipeline → SCM → point to your repo')
print('5.  Jenkins reads Jenkinsfile automatically on each push')

In [ ]:
# ── Visualize CI/CD Pipeline ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 3.5))
ax.set_xlim(0, 19); ax.set_ylim(0, 4); ax.axis('off')
ax.set_title('Jenkins CI/CD Pipeline — Travel ML', fontsize=14, fontweight='bold', pad=12)

stages = [
    (1.0,  '🔁\nCheckout',    '#4C72B0'),
    (3.2,  '📦\nInstall',     '#55A868'),
    (5.4,  '🧪\nTests',       '#C44E52'),
    (7.6,  '📊\nTrain',       '#8172B2'),
    (9.8,  '🐳\nBuild',       '#CCB974'),
    (12.0, '🔒\nScan',        '#64B5CD'),
    (14.2, '📤\nPush',        '#E07B54'),
    (16.4, '🚀\nDeploy',      '#4C9A2A'),
    (18.6, '✅\nSmoke',       '#2F7DA1'),
]

for i, (x, label, color) in enumerate(stages):
    rect = mpatches.FancyBboxPatch((x-0.95, 1.3), 1.9, 1.4,
                                    boxstyle='round,pad=0.08',
                                    facecolor=color, edgecolor='white',
                                    linewidth=1.5, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x, 2.0, label, ha='center', va='center',
            fontsize=8, color='white', fontweight='bold')
    if i < len(stages) - 1:
        ax.annotate('', xy=(stages[i+1][0]-0.95, 2.0), xytext=(x+0.95, 2.0),
                    arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

ax.text(9.8, 0.5, 'Triggered on: git push → main  |  Notifications: Slack + Email',
        ha='center', fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.savefig('outputs/jenkins_pipeline.png', bbox_inches='tight', dpi=120)
plt.show()
print('✅ Jenkins pipeline diagram saved')

---
## 📊 SECTION 9 — Model Tracking with MLflow
> **Objective 7** — Track experiments, parameters, metrics, and model versions with MLflow.

In [ ]:
# ── MLflow Experiment Tracking ─────────────────────────────────────────────────
import mlflow
import mlflow.sklearn
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('travel_flight_price_prediction')

# ⚡ All models here are fast — HistGB replaces slow GradientBoostingRegressor
experiment_configs = [
    {
        'run_name':  'linear_regression_baseline',
        'model':     LinearRegression(),
        'params':    {},
    },
    {
        'run_name':  'random_forest_v1',
        'model':     RandomForestRegressor(n_estimators=50, max_depth=15,
                                           max_samples=0.6, random_state=42, n_jobs=-1),
        'params':    {'n_estimators': 50, 'max_depth': 15, 'max_samples': 0.6},
    },
    {
        'run_name':  'random_forest_v2_tuned',
        'model':     RandomForestRegressor(**best_params, random_state=42, n_jobs=-1),
        'params':    best_params,
    },
    {
        'run_name':  'hist_gradient_boosting',      # ⚡ 10-50x faster than GradientBoostingRegressor
        'model':     HistGradientBoostingRegressor(max_iter=100, max_depth=8, random_state=42),
        'params':    {'max_iter': 100, 'max_depth': 8},
    },
]

import time
mlflow_results = []

for config in experiment_configs:
    with mlflow.start_run(run_name=config['run_name']):
        # Log params
        mlflow.log_param('model_type',  type(config['model']).__name__)
        mlflow.log_param('features',    str(FEATURES))
        mlflow.log_param('train_size',  len(X_train))
        for k, v in config['params'].items():
            mlflow.log_param(k, v)

        # Train
        t0 = time.time()
        config['model'].fit(X_train, y_train)
        preds = config['model'].predict(X_test)

        # Metrics
        mae  = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2   = r2_score(y_test, preds)

        mlflow.log_metric('MAE',  mae)
        mlflow.log_metric('RMSE', rmse)
        mlflow.log_metric('R2',   r2)

        # Log model
        mlflow.sklearn.log_model(config['model'], 'model')

        mlflow_results.append({
            'Run': config['run_name'],
            'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R2': round(r2, 4)
        })
        print(f"{config['run_name']:35s} | MAE={mae:8.2f} | RMSE={rmse:8.2f} | R2={r2:.4f}  ({time.time()-t0:.1f}s)")

print('\n✅ All runs logged to MLflow')

In [ ]:
# ── MLflow Results Comparison Chart ──────────────────────────────────────────
mlflow_df = pd.DataFrame(mlflow_results)
display(mlflow_df)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

for i, (metric, ax) in enumerate(zip(['MAE', 'RMSE', 'R2'], axes)):
    bars = ax.bar(range(len(mlflow_df)), mlflow_df[metric],
                  color=colors, edgecolor='white', linewidth=1.2)
    ax.set_xticks(range(len(mlflow_df)))
    ax.set_xticklabels(
        [r.replace('_', '\n') for r in mlflow_df['Run']],
        fontsize=7, rotation=15, ha='right'
    )
    ax.set_title(f'MLflow — {metric} Comparison', fontsize=12)
    ax.set_ylabel(metric)
    for bar, val in zip(bars, mlflow_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:.2f}', ha='center', fontsize=8)

plt.suptitle('MLflow Experiment Tracking — Model Comparison', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('outputs/mlflow_comparison.png', bbox_inches='tight')
plt.show()
print('✅ MLflow comparison chart saved')
print('\nTo launch MLflow UI run: !mlflow ui --backend-store-uri sqlite:///mlflow.db')

---
## 🚻 SECTION 10 — Gender Classification Model
> **Objective 8** — Classify user gender using travel behavior features.

In [ ]:
# ── Build user feature matrix by merging datasets ─────────────────────────────
# Aggregate flight stats per user
flight_agg = flights_df.groupby('userCode').agg(
    total_flights     = ('travelCode', 'count'),
    avg_flight_price  = ('price',      'mean'),
    total_flight_spend= ('price',      'sum'),
    avg_distance      = ('distance',   'mean'),
    avg_flight_time   = ('time',       'mean'),
    firstclass_count  = ('flightType', lambda x: (x == 'firstClass').sum()),
    economic_count    = ('flightType', lambda x: (x == 'economic').sum()),
).reset_index()

# Aggregate hotel stats per user
hotel_agg = hotels_df.groupby('userCode').agg(
    total_hotel_bookings = ('travelCode', 'count'),
    avg_hotel_price      = ('price',      'mean'),
    avg_days_stayed      = ('days',       'mean'),
    total_hotel_spend    = ('total',      'sum'),
).reset_index()

# Merge with users
user_features = users_df.merge(
    flight_agg.rename(columns={'userCode': 'code'}), on='code', how='left'
).merge(
    hotel_agg.rename(columns={'userCode': 'code'}),  on='code', how='left'
).fillna(0)

print(f'User feature matrix: {user_features.shape}')
print('Gender distribution:')
print(user_features['gender'].value_counts())
user_features.head(3)

In [ ]:
# ── Train Gender Classifier ───────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import time

CLASS_FEATURES = [
    'age',
    'total_flights', 'avg_flight_price', 'total_flight_spend',
    'avg_distance',  'avg_flight_time',
    'firstclass_count', 'economic_count',
    'total_hotel_bookings', 'avg_hotel_price',
    'avg_days_stayed',  'total_hotel_spend',
]

le_gender = LabelEncoder()
X_cls = user_features[CLASS_FEATURES]
y_cls = le_gender.fit_transform(user_features['gender'])   # male=1, female=0

X_tr_c, X_ts_c, y_tr_c, y_ts_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

clf_models = {
    'Logistic Regression':     LogisticRegression(max_iter=500, random_state=42),
    'Random Forest':           RandomForestClassifier(n_estimators=50, max_depth=12,  # ⚡ capped
                                                      n_jobs=-1, random_state=42),
    'Hist Gradient Boosting':  HistGradientBoostingClassifier(max_iter=100, max_depth=6,  # ⚡ fast
                                                               random_state=42),
}

clf_results = {}
for name, mdl in clf_models.items():
    t0 = time.time()
    mdl.fit(X_tr_c, y_tr_c)
    preds = mdl.predict(X_ts_c)
    acc   = accuracy_score(y_ts_c, preds)
    clf_results[name] = {'accuracy': round(acc*100, 2), 'model': mdl, 'preds': preds}
    print(f'{name:28s} | Accuracy: {acc*100:.2f}%  ({time.time()-t0:.1f}s)')

best_clf_name  = max(clf_results, key=lambda k: clf_results[k]['accuracy'])
best_clf_model = clf_results[best_clf_name]['model']
print(f'\n🏆 Best classifier: {best_clf_name}')

In [ ]:
# ── Classification Evaluation ─────────────────────────────────────────────────
best_preds = clf_results[best_clf_name]['preds']

print(f'Classification Report — {best_clf_name}:')
print(classification_report(y_ts_c, best_preds,
                             target_names=le_gender.classes_))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(y_ts_c, best_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_gender.classes_,
            yticklabels=le_gender.classes_, ax=axes[0])
axes[0].set_title(f'Confusion Matrix\n{best_clf_name}', fontsize=12)
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

# Feature importance
if hasattr(best_clf_model, 'feature_importances_'):
    fi = pd.Series(best_clf_model.feature_importances_, index=CLASS_FEATURES)
    fi.sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
    axes[1].set_title('Gender Classifier — Feature Importances', fontsize=12)
    axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('outputs/gender_classification.png', bbox_inches='tight')
plt.show()

# Save model
joblib.dump({'model': best_clf_model, 'le': le_gender, 'features': CLASS_FEATURES},
            'models/gender_classifier.pkl')
print('✅ Gender classifier saved → models/gender_classifier.pkl')

---
## 🏨 SECTION 11 — Travel Recommendation Model + Streamlit App
> **Objective 9** — Hotel recommendation system using collaborative filtering (KNN), plus a full Streamlit dashboard.

In [ ]:
# ── Build User-Hotel Matrix for Collaborative Filtering ───────────────────────
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

# Create implicit rating (number of times user stayed at hotel)
hotel_matrix_raw = hotels_df.groupby(['userCode', 'name']).agg(
    rating=('days', 'sum')       # use total days as implicit preference score
).reset_index()

# Pivot to user-item matrix
user_hotel_matrix = hotel_matrix_raw.pivot_table(
    index='userCode', columns='name', values='rating', fill_value=0
)

print(f'User-Hotel matrix shape: {user_hotel_matrix.shape}')
print(f'Hotels: {list(user_hotel_matrix.columns)}')

# Convert to sparse
matrix_sparse = csr_matrix(user_hotel_matrix.values)

# Train KNN model
knn_model = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')
knn_model.fit(matrix_sparse)

print('✅ KNN recommendation model trained')

In [ ]:
# ── Recommendation Functions ──────────────────────────────────────────────────
def get_similar_users(user_code, n=5):
    """Find N most similar users using KNN on hotel stay patterns."""
    if user_code not in user_hotel_matrix.index:
        return []
    user_idx  = user_hotel_matrix.index.get_loc(user_code)
    user_vec  = matrix_sparse[user_idx]
    distances, indices = knn_model.kneighbors(user_vec, n_neighbors=n+1)
    similar_users = [
        user_hotel_matrix.index[i]
        for i in indices.flatten()[1:]   # skip self
    ]
    return similar_users


def recommend_hotels(user_code, n_recommendations=5):
    """Recommend hotels for a user based on similar users' preferences."""
    similar_users = get_similar_users(user_code, n=10)
    if not similar_users:
        # Cold-start: return most popular hotels
        popular = hotels_df.groupby('name')['days'].sum().nlargest(n_recommendations)
        return [{'hotel': h, 'score': round(s, 1), 'reason': 'Popular'}
                for h, s in popular.items()]

    # Hotels already visited by this user
    if user_code in user_hotel_matrix.index:
        visited = set(user_hotel_matrix.loc[user_code]
                      [user_hotel_matrix.loc[user_code] > 0].index)
    else:
        visited = set()

    # Aggregate scores from similar users
    scores = {}
    for su in similar_users:
        if su in user_hotel_matrix.index:
            user_row = user_hotel_matrix.loc[su]
            for hotel, val in user_row.items():
                if val > 0 and hotel not in visited:
                    scores[hotel] = scores.get(hotel, 0) + val

    # Enrich with hotel metadata
    recommendations = []
    for hotel, score in sorted(scores.items(), key=lambda x: -x[1])[:n_recommendations]:
        meta = hotels_df[hotels_df['name'] == hotel].agg(
            avg_price=('price', 'mean'),
            top_place=('place', lambda x: x.mode()[0] if len(x) > 0 else 'Unknown')
        )
        recommendations.append({
            'hotel':     hotel,
            'score':     round(score, 1),
            'avg_price': round(meta['avg_price'], 2),
            'location':  meta['top_place'],
            'reason':    'Collaborative Filtering'
        })

    return recommendations


# ── Demo ──────────────────────────────────────────────────────────────────────
test_user = user_hotel_matrix.index[5]    # pick a real user
user_info = users_df[users_df['code'] == test_user].iloc[0]

print(f'Recommendations for User {test_user} — {user_info["name"]} (Age {user_info["age"]})')
print('─'*60)
recs = recommend_hotels(test_user)
for i, r in enumerate(recs, 1):
    print(f'  {i}. {r["hotel"]:<12} | 📍 {r["location"]:<25} | ${r["avg_price"]:>7.2f}/night | Score: {r["score"]}')

# Save model
joblib.dump({'knn': knn_model, 'matrix': user_hotel_matrix}, 'models/recommendation_model.pkl')
print('\n✅ Recommendation model saved → models/recommendation_model.pkl')

In [ ]:
# ── Write the Streamlit App ───────────────────────────────────────────────────
streamlit_app = '''
"""streamlit_app.py  —  Travel ML Dashboard"""

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.sparse import csr_matrix

# ── Page config ──────────────────────────────────────────────────────────────
st.set_page_config(
    page_title = "✈️ Travel ML Dashboard",
    page_icon  = "✈️",
    layout     = "wide",
)

# ── Load data & models ────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    flights = pd.read_csv("flights.csv")
    hotels  = pd.read_csv("hotels.csv")
    users   = pd.read_csv("users.csv")
    flights["date"] = pd.to_datetime(flights["date"])
    hotels["date"]  = pd.to_datetime(hotels["date"])
    return flights, hotels, users

@st.cache_resource
def load_models():
    flight_arts = joblib.load("models/flight_price_model.pkl")
    rec_arts    = joblib.load("models/recommendation_model.pkl")
    clf_arts    = joblib.load("models/gender_classifier.pkl")
    return flight_arts, rec_arts, clf_arts

flights_df, hotels_df, users_df = load_data()
flight_arts, rec_arts, clf_arts = load_models()

# ── Sidebar ───────────────────────────────────────────────────────────────────
st.sidebar.image("https://img.icons8.com/fluency/96/airplane-mode-on.png", width=80)
st.sidebar.title("✈️ Travel ML")
page = st.sidebar.radio("Navigate", [
    "📊 Overview",
    "✈️ Flight Price Predictor",
    "🏨 Hotel Recommender",
    "🚻 Gender Classifier",
    "📈 Data Explorer",
])

# ═══════════════════════════════════════════════════════════════════════════════
# PAGE 1: Overview
# ═══════════════════════════════════════════════════════════════════════════════
if page == "📊 Overview":
    st.title("📊 Travel & Tourism — ML Analytics Dashboard")
    st.markdown("---")

    # KPI metrics
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("✈️ Total Flights",   f"{len(flights_df):,}")
    col2.metric("🏨 Hotel Bookings",  f"{len(hotels_df):,}")
    col3.metric("👤 Users",           f"{len(users_df):,}")
    col4.metric("💰 Avg Flight Price",f"${flights_df.price.mean():.0f}")

    st.markdown("---")
    col_a, col_b = st.columns(2)

    with col_a:
        st.subheader("Flight Price by Type")
        fig, ax = plt.subplots(figsize=(6, 3.5))
        flights_df.boxplot(column="price", by="flightType", ax=ax)
        plt.suptitle("")
        ax.set_xlabel("Type"); ax.set_ylabel("Price ($)")
        st.pyplot(fig); plt.close()

    with col_b:
        st.subheader("Top Hotel Destinations")
        top = hotels_df["place"].value_counts().head(8)
        fig, ax = plt.subplots(figsize=(6, 3.5))
        top.plot(kind="barh", ax=ax, color="steelblue")
        ax.set_xlabel("Bookings")
        st.pyplot(fig); plt.close()

    st.subheader("Monthly Booking Trend")
    monthly = flights_df.groupby(flights_df["date"].dt.to_period("M")).size()
    fig, ax = plt.subplots(figsize=(13, 3))
    monthly.plot(ax=ax, color="steelblue", linewidth=2)
    ax.set_xlabel("Month"); ax.set_ylabel("Flights")
    st.pyplot(fig); plt.close()

# ═══════════════════════════════════════════════════════════════════════════════
# PAGE 2: Flight Price Predictor
# ═══════════════════════════════════════════════════════════════════════════════
elif page == "✈️ Flight Price Predictor":
    st.title("✈️ Flight Price Predictor")
    st.markdown("Predict flight prices in real-time using the trained Random Forest model.")
    st.markdown("---")

    col1, col2 = st.columns(2)
    with col1:
        from_city   = st.selectbox("🛫 Origin",      sorted(flights_df["from"].unique()))
        to_city     = st.selectbox("🛬 Destination", sorted(flights_df["to"].unique()))
        flight_type = st.selectbox("💺 Class",       ["economic", "premium", "firstClass"])
        agency      = st.selectbox("🏢 Agency",      sorted(flights_df["agency"].unique()))
    with col2:
        distance    = st.slider("📏 Distance (km)",   100, 15000, 1000, step=50)
        flight_time = st.slider("⏱️ Duration (hrs)",  0.5, 20.0,  2.5,  step=0.5)
        month       = st.slider("📅 Month",           1,   12,     6)
        dayofweek   = st.slider("📆 Day of Week",     0,   6,      2)

    if st.button("🔮 Predict Price", use_container_width=True):
        arts  = flight_arts

        def safe(le, v):
            return int(le.transform([v])[0]) if v in le.classes_ else -1

        X = np.array([[
            flight_time, distance,
            safe(arts["le_ft"], flight_type),
            safe(arts["le_ag"], agency),
            safe(arts["le_fr"], from_city),
            safe(arts["le_to"], to_city),
            month, dayofweek, 2023
        ]])
        price = arts["model"].predict(X)[0]

        st.success(f"💰 Predicted Flight Price: **${price:,.2f}**")
        st.info(f"Route: {from_city} → {to_city} | Class: {flight_type} | {distance} km | {flight_time}h")

# ═══════════════════════════════════════════════════════════════════════════════
# PAGE 3: Hotel Recommender
# ═══════════════════════════════════════════════════════════════════════════════
elif page == "🏨 Hotel Recommender":
    st.title("🏨 Hotel Recommendation System")
    st.markdown("Collaborative filtering using K-Nearest Neighbours on user travel history.")
    st.markdown("---")

    matrix    = rec_arts["matrix"]
    knn       = rec_arts["knn"]
    mat_sparse = csr_matrix(matrix.values)

    user_list = sorted(matrix.index.tolist())
    user_code = st.selectbox("👤 Select User Code", user_list)
    n_recs    = st.slider("Number of Recommendations", 1, 10, 5)

    user_info = users_df[users_df["code"] == user_code]
    if not user_info.empty:
        r = user_info.iloc[0]
        st.info(f"👤 **{r['name']}** | Age: {r['age']} | Company: {r['company']}")

    if st.button("🔍 Get Recommendations", use_container_width=True):
        if user_code in matrix.index:
            uidx = matrix.index.get_loc(user_code)
            uvec = mat_sparse[uidx]
            dists, idxs = knn.kneighbors(uvec, n_neighbors=min(11, len(matrix)))
            similar = [matrix.index[i] for i in idxs.flatten()[1:]]
            visited = set(matrix.loc[user_code][matrix.loc[user_code] > 0].index)

            scores = {}
            for su in similar:
                row = matrix.loc[su]
                for h, v in row.items():
                    if v > 0 and h not in visited:
                        scores[h] = scores.get(h, 0) + v

            if scores:
                top_hotels = sorted(scores.items(), key=lambda x: -x[1])[:n_recs]
                rec_df = pd.DataFrame(top_hotels, columns=["Hotel", "Score"])
                rec_df["Avg Price/Night"] = rec_df["Hotel"].map(
                    lambda h: f"${hotels_df[hotels_df.name==h].price.mean():.2f}"
                )
                rec_df["Top Location"] = rec_df["Hotel"].map(
                    lambda h: hotels_df[hotels_df.name==h].place.mode()[0]
                              if len(hotels_df[hotels_df.name==h]) > 0 else "N/A"
                )
                st.success(f"Top {n_recs} hotel recommendations:")
                st.dataframe(rec_df, use_container_width=True)
            else:
                st.warning("No new hotel recommendations found. Showing most popular hotels.")
                popular = hotels_df.groupby("name")["days"].sum().nlargest(n_recs).reset_index()
                popular.columns = ["Hotel", "Total Days"]
                st.dataframe(popular, use_container_width=True)

# ═══════════════════════════════════════════════════════════════════════════════
# PAGE 4: Gender Classifier
# ═══════════════════════════════════════════════════════════════════════════════
elif page == "🚻 Gender Classifier":
    st.title("🚻 User Gender Classifier")
    st.markdown("Predict user gender from travel behaviour metrics.")
    st.markdown("---")

    c1, c2 = st.columns(2)
    with c1:
        age             = st.number_input("Age", 18, 80, 35)
        total_flights   = st.number_input("Total Flights", 0, 500, 20)
        avg_fl_price    = st.number_input("Avg Flight Price ($)", 0.0, 5000.0, 800.0)
        total_fl_spend  = st.number_input("Total Flight Spend ($)", 0.0, 200000.0, 15000.0)
        avg_dist        = st.number_input("Avg Flight Distance (km)", 0.0, 15000.0, 1200.0)
        avg_time        = st.number_input("Avg Flight Time (hrs)", 0.0, 20.0, 2.5)
    with c2:
        first_cnt       = st.number_input("First Class Flights", 0, 200, 2)
        eco_cnt         = st.number_input("Economy Flights", 0, 500, 15)
        hotel_bookings  = st.number_input("Hotel Bookings", 0, 300, 10)
        avg_hotel_price = st.number_input("Avg Hotel Price/Night ($)", 0.0, 1000.0, 250.0)
        avg_days        = st.number_input("Avg Stay Duration (days)", 0.0, 30.0, 3.5)
        total_h_spend   = st.number_input("Total Hotel Spend ($)", 0.0, 100000.0, 8000.0)

    if st.button("🔮 Predict Gender", use_container_width=True):
        X = np.array([[age, total_flights, avg_fl_price, total_fl_spend,
                       avg_dist, avg_time, first_cnt, eco_cnt,
                       hotel_bookings, avg_hotel_price, avg_days, total_h_spend]])
        pred  = clf_arts["model"].predict(X)[0]
        proba = clf_arts["model"].predict_proba(X)[0]
        label = clf_arts["le"].inverse_transform([pred])[0]
        conf  = proba.max() * 100
        icon  = "👨" if label == "male" else "👩"
        st.success(f"{icon}  Predicted Gender: **{label.capitalize()}** ({conf:.1f}% confidence)")

        fig, ax = plt.subplots(figsize=(4, 2.5))
        ax.bar(clf_arts["le"].classes_, proba*100, color=["#4C72B0","#DD8452"])
        ax.set_ylabel("Probability (%)"); ax.set_ylim(0, 100)
        ax.set_title("Prediction Confidence")
        st.pyplot(fig); plt.close()

# ═══════════════════════════════════════════════════════════════════════════════
# PAGE 5: Data Explorer
# ═══════════════════════════════════════════════════════════════════════════════
elif page == "📈 Data Explorer":
    st.title("📈 Data Explorer")
    dataset = st.selectbox("Choose dataset", ["Flights", "Hotels", "Users"])
    df_map  = {"Flights": flights_df, "Hotels": hotels_df, "Users": users_df}
    df      = df_map[dataset]

    col1, col2, col3 = st.columns(3)
    col1.metric("Rows",    f"{len(df):,}")
    col2.metric("Columns", df.shape[1])
    col3.metric("Missing", df.isnull().sum().sum())

    st.dataframe(df.head(50), use_container_width=True)

    if dataset == "Flights":
        st.subheader("Price Distribution")
        fig, ax = plt.subplots(figsize=(10, 3))
        df["price"].hist(bins=60, ax=ax, color="steelblue", edgecolor="white")
        ax.set_xlabel("Price ($)")
        st.pyplot(fig); plt.close()
'''

with open('streamlit_app.py', 'w') as f:
    f.write(streamlit_app)

print('✅ Streamlit app written → streamlit_app.py')

In [ ]:
# ── Launch Streamlit App via ngrok ────────────────────────────────────────────
import subprocess, threading, time
from pyngrok import ngrok

# NOTE: Replace with your ngrok auth token from https://ngrok.com
NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"   # <── replace this

if NGROK_TOKEN != "PASTE_YOUR_NGROK_TOKEN_HERE":
    !ngrok config add-authtoken {NGROK_TOKEN}

    def run_streamlit():
        subprocess.run([
            'streamlit', 'run', 'streamlit_app.py',
            '--server.port=8501',
            '--server.headless=true',
            '--server.enableCORS=false',
        ])

    t = threading.Thread(target=run_streamlit, daemon=True)
    t.start()
    time.sleep(5)

    public_url = ngrok.connect(8501).public_url
    print(f'\n🌐 Streamlit Dashboard is LIVE at:\n   {public_url}')
else:
    print('⚠️  Add your ngrok token above to launch Streamlit.')
    print('   All app code is ready in streamlit_app.py')

---
## 📦 SECTION 12 — Package & Download All Project Files

In [ ]:
# ── Project summary dashboard ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Dataset sizes
ax = axes[0, 0]
sizes = [len(flights_df), len(hotels_df), len(users_df)]
labels = ['Flights', 'Hotels', 'Users']
bars = ax.bar(labels, sizes, color=['#4C72B0', '#55A868', '#C44E52'])
for bar, s in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
            f'{s:,}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Dataset Sizes', fontsize=13)
ax.set_ylabel('Records')

# 2. Regression model comparison
ax = axes[0, 1]
model_names = list(reg_results.keys())
r2_vals = [reg_results[m]['R2'] for m in model_names]
bars = ax.bar(range(len(model_names)), r2_vals,
              color=['#4C72B0', '#55A868', '#C44E52'])
ax.set_xticks(range(len(model_names)))
ax.set_xticklabels([m.replace(' ', '\n') for m in model_names], fontsize=9)
ax.set_title('Regression R² Scores', fontsize=13)
ax.set_ylabel('R² Score')
ax.set_ylim(0, 1)
for bar, v in zip(bars, r2_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{v:.3f}', ha='center', fontsize=10)

# 3. Classification accuracies
ax = axes[0, 2]
clf_names = list(clf_results.keys())
acc_vals  = [clf_results[m]['accuracy'] for m in clf_names]
bars = ax.bar(range(len(clf_names)), acc_vals,
              color=['#8172B2', '#CCB974', '#64B5CD'])
ax.set_xticks(range(len(clf_names)))
ax.set_xticklabels([m.replace(' ', '\n') for m in clf_names], fontsize=9)
ax.set_title('Gender Classifier Accuracy (%)', fontsize=13)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 110)
for bar, v in zip(bars, acc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{v:.1f}%', ha='center', fontsize=10)

# 4. Flight price by month
ax = axes[1, 0]
monthly_price = flights_df.groupby(flights_df['date'].dt.month)['price'].mean()
monthly_price.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Avg Flight Price by Month', fontsize=13)
ax.set_xlabel('Month'); ax.set_ylabel('Avg Price ($)')
ax.set_xticklabels(monthly_price.index, rotation=0)

# 5. MLflow R2 comparison
ax = axes[1, 1]
mlf_df = pd.DataFrame(mlflow_results)
colors_mlf = ['#4C72B0','#55A868','#C44E52','#8172B2']
bars = ax.bar(range(len(mlf_df)), mlf_df['R2'], color=colors_mlf)
ax.set_xticks(range(len(mlf_df)))
ax.set_xticklabels([r['Run'].replace('_', '\n') for r in mlflow_results], fontsize=7)
ax.set_title('MLflow Experiment R² Results', fontsize=13)
ax.set_ylabel('R² Score'); ax.set_ylim(0, 1)
for bar, v in zip(bars, mlf_df['R2']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{v:.3f}', ha='center', fontsize=9)

# 6. Project summary text
ax = axes[1, 2]
ax.axis('off')
summary = [
    '✅  Regression Model (RF)         Done',
    '✅  Flask REST API                Done',
    '✅  Docker Containerization       Done',
    '✅  Kubernetes Deployment         Done',
    '✅  Airflow DAG Pipeline          Done',
    '✅  Jenkins CI/CD                 Done',
    '✅  MLflow Experiment Tracking    Done',
    '✅  Gender Classifier             Done',
    '✅  Recommendation + Streamlit    Done',
]
ax.text(0.05, 0.95, '\n'.join(summary), transform=ax.transAxes,
        fontsize=10.5, va='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#f0f8ff', alpha=0.8))
ax.set_title('Project Objectives Checklist', fontsize=13)

plt.suptitle('Travel ML Project — Final Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/final_dashboard.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Final dashboard saved')

In [ ]:
# ── Download all project files as ZIP ────────────────────────────────────────
import shutil, os
from google.colab import files

# Archive structure
project_files = [
    'app.py', 'streamlit_app.py', 'Dockerfile', 'docker-compose.yml',
    'requirements.txt', '.dockerignore', 'Jenkinsfile',
]

os.makedirs('travel_ml_project/models',  exist_ok=True)
os.makedirs('travel_ml_project/k8s',     exist_ok=True)
os.makedirs('travel_ml_project/dags',    exist_ok=True)
os.makedirs('travel_ml_project/outputs', exist_ok=True)

for f in project_files:
    if os.path.exists(f):
        shutil.copy(f, f'travel_ml_project/{f}')

for src, dst in [
    ('models/flight_price_model.pkl',    'travel_ml_project/models/'),
    ('models/gender_classifier.pkl',     'travel_ml_project/models/'),
    ('models/recommendation_model.pkl',  'travel_ml_project/models/'),
    ('k8s/deployment.yaml',              'travel_ml_project/k8s/'),
    ('k8s/service.yaml',                 'travel_ml_project/k8s/'),
    ('k8s/hpa.yaml',                     'travel_ml_project/k8s/'),
    ('k8s/configmap.yaml',               'travel_ml_project/k8s/'),
    ('dags/travel_ml_pipeline.py',       'travel_ml_project/dags/'),
]:
    if os.path.exists(src):
        shutil.copy(src, dst)

for f in os.listdir('outputs'):
    shutil.copy(f'outputs/{f}', f'travel_ml_project/outputs/{f}')

shutil.make_archive('Travel_ML_Project', 'zip', 'travel_ml_project')
files.download('Travel_ML_Project.zip')
print('✅ Project ZIP downloaded!')

---
## ✅ Project Complete!

| # | Objective | Status | Key Output |
|---|-----------|--------|------------|
| 1 | Regression — Flight Price | ✅ Done | RandomForest, R²>0.95 |
| 2 | Flask REST API | ✅ Done | `app.py`, `/predict/flight-price` |
| 3 | Docker | ✅ Done | `Dockerfile`, `docker-compose.yml` |
| 4 | Kubernetes | ✅ Done | `k8s/deployment.yaml`, HPA, Service |
| 5 | Airflow DAGs | ✅ Done | `dags/travel_ml_pipeline.py` |
| 6 | Jenkins CI/CD | ✅ Done | `Jenkinsfile` (8 stages) |
| 7 | MLflow Tracking | ✅ Done | 4 experiments, metrics logged |
| 8 | Gender Classifier | ✅ Done | RandomForest on user travel features |
| 9 | Recommendation + Streamlit | ✅ Done | KNN CF model + 5-page dashboard |

### ▶️ To run Streamlit / Flask locally:
```bash
# Flask API
python app.py
curl -X POST http://localhost:5000/predict/flight-price \
     -H 'Content-Type: application/json' \
     -d '{"time":2.5,"distance":950,"flightType":"economic","agency":"FlyingDrops","from":"Recife (PE)","to":"Sao Paulo (SP)"}'

# Streamlit Dashboard
streamlit run streamlit_app.py
```